In [ ]:
import PIL.Image as Image
import torch
from torchvision.io import video_reader

In [ ]:
reader = video_reader.VideoReader(
    "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/ERA_UAV_Video_Dataset/Videos/Test/Baseball/Baseball_001 .mp4",
    num_threads=2,
)
m = reader.get_metadata()
print(m)


# one frame
for frame in reader:
    print(frame["pts"], frame["data"].shape)
    data = frame["data"].numpy().transpose(1, 2, 0)  # HWC
    img = Image.fromarray(data)
    break

img

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 存储要展示的帧和对应时间戳
frames = []
timestamps = []

for frame in reader:
    pts = frame["pts"]

    # 每隔0.5秒取一帧
    if pts % 0.5 <= 0.001:  # 避免浮点精度问题
        data = frame["data"].numpy().transpose(1, 2, 0)  # (H, W, C)
        img = Image.fromarray(data)
        frames.append(img)
        timestamps.append(pts)

    # 可选：避免读取太多帧
    if pts > 10:  # 最多读取到10秒（可选）
        break

# 在Matplotlib中绘制
plt.figure(figsize=(12, 8))
n_frames = len(frames)

for i, (img, ts) in enumerate(zip(frames, timestamps)):
    plt.subplot((n_frames // 4) + 1, 4, i + 1)  # 网格排列，每行最多4张
    plt.imshow(img)
    plt.title(f"pts={ts:.2f}s")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [11]:
from torchcodec.decoders import VideoDecoder

reader = VideoDecoder(
    "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/ERA_UAV_Video_Dataset/Videos/Test/Baseball/Baseball_022 .mp4",
    device="cpu",
)

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0.dev20250424+cu126) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 7: libavutil.so.59: cannot open shared object file: No such file or directory
FFmpeg version 6: libavutil.so.58: cannot open shared object file: No such file or directory
FFmpeg version 5: libavutil.so.57: cannot open shared object file: No such file or directory
FFmpeg version 4: /lib/x86_64-linux-gnu/libgobject-2.0.so.0: undefined symbol: ff

In [9]:
import torch

print(torch.__version__)  # 需与 torchcodec 要求的版本一致
print(torch.cuda.is_available())  # 必须返回 True（因指定了 device="cuda"）

2.8.0.dev20250424+cu126
True


In [12]:
!ffmpeg -decoders | grep -i nvidia

ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --e

In [14]:
!ffmpeg -hwaccel cuda  -hwaccel_output_format cuda -i "/Data4/cao/ZiHanCao/exps/HyperspectralTokenizer/data/ERA_UAV_Video_Dataset/Videos/Test/Baseball/Baseball_001 .mp4" -f null -

ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --e